### Imports ###

In [ ]:
!pip install psycopg2

In [ ]:
import pandas as pd
import numpy as np
import psycopg2.extras as extras

### Load Configs ###

In [ ]:
import sys
sys.path.append("..")

from config import get_connection, logger

### ETL Pipeline ###

#### Extract (E) - Process ####

In [ ]:
logger.info("Starting Jobs ETL - Extract")

CSV_FILE = "../raw_data/jobs.csv"
df = pd.read_csv(CSV_FILE)

logger.info(f"Extracted {len(df)} job records")
display(df.head())

#### Transform (T) - Process ####

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

df["job_id"] = df["job_id"].astype(str)
df["company_id"] = df["company_id"].astype(int)

df = df.drop_duplicates(subset = ["job_id"])

#### Load (L) - Process ####

In [ ]:
query = """
INSERT INTO jobs (job_id, title, job_description, company_id)
VALUES %s
ON CONFLICT (job_id)
DO UPDATE SET
title = EXCLUDED.title,
job_description = EXCLUDED.job_description,
company_id = EXCLUDED.company_id
"""

conn = get_connection()
with conn.cursor() as cur:
    extras.execute_values(
        cur,
        query,
        df[['job_id', 'title', 'job_description', 'company_id']].values.tolist()
    )
    conn.commit()
conn.close()

logger.info("Jobs ETL completed")